# M02 — Ridge, lasso, and logistic models are different commitments

<!-- paper-first -->
### Begin with the paper question

**Read or revisit [PM01](../../curriculum/papers/modeling.md#pm01), [PM03](../../curriculum/papers/modeling.md#pm03).** Use the assigned first-pass sections in the guide; if you already read them, return only to the relevant figure or claim. Do this before the technical explanation below.

**Motivation:** What scientific comparison would a regularized linear baseline make possible?

Write a two-sentence prediction and one thing you cannot yet explain. Ask your AI tutor to locate evidence in the supplied paper and distinguish it from inference. A paper link motivates this question; it does not mean the paper uses every method demonstrated here.

**After the experiment:** revisit your prediction in the [evidence ledger](../../curriculum/coursework/EVIDENCE_LEDGER.md). Explain one mechanism you now understand, cite a result from this notebook, and name a paper claim this exercise still cannot test. Keep a small demonstration distinct from a reproduction of the study.
<!-- /paper-first -->

**Original guided lab · 75–100 minutes.** Read 20 min, predict/code 35 min, failure investigation 20 min, explain and transfer 15 min. Run all cells in order in a fresh kernel. All executed data are synthetic unless explicitly stated. No network, GPU, or external dataset is required.

Linear models are useful scientific baselines because we can inspect their input contract, fitting objective, and failure cases. Linear does not mean only one feature: a weighted combination can contain many voxels, regional measures, nuisance columns, or stimulus descriptors. A model remains linear in its fitted coefficients even if some input columns represent nonlinear transformations chosen beforehand.

Ordinary least squares minimizes squared residuals. When there are many correlated imaging features, many coefficient combinations can fit the training data similarly. Ridge adds a squared-coefficient penalty and shrinks weights toward zero. Lasso adds an absolute-coefficient penalty and can produce exact zeros. Sparse coefficients are an output of the objective and feature correlations, not proof that all omitted regions are irrelevant. If two nearly interchangeable features carry the same signal, lasso may favor one while ridge shares weight between them.

Penalty strength is a hyperparameter. Its numerical value depends on the library's objective scaling, sample size conventions, and feature units. Standardization must therefore be fitted on training observations and applied using those same constants. The code uses pipelines to keep scaling together with the estimator. We compare fixed demonstration strengths here; a research selection of strength belongs inside inner validation folds, as in notebook 04. Coefficients from differently scaled representations cannot be compared without translating their units.

For classification, logistic regression models a probability through a sigmoid applied to a linear score. It uses a likelihood appropriate to binary labels, rather than assuming a Gaussian continuous outcome. A threshold turns probability into a category and introduces a separate decision rule. It is possible for a model to improve ranking without having well-calibrated probabilities, which is why notebook 18 revisits calibration. Class imbalance also makes a constant-class baseline essential.

The experiment intentionally has more columns than training participants. Its independent holdout reveals the difference between fitting and generalizing. A deliberately excessive ridge penalty approaches a constant prediction; shrinking is useful only in relation to the signal and the objective. An AI assistant should not say that regularization always improves a score. It should show which loss changed, how much the coefficients moved, and how the comparison was evaluated. Keep both successful and unsuccessful comparisons so that the final explanation includes evidence rather than only the winning method.

## Transformation contract

Training rows → train-fitted standardization → regularized weights → held-out number or probability. Scaling changes units; shrinkage changes the fitted relationship; a linear prediction compresses a feature vector to an outcome.

## Ask your AI tutor

```text
Explain this notebook one transformation at a time.
Before each cell ask me to predict shapes, units, and a check.
Give edits in executable cells of at most 20 lines.
Keep the prescribed split, random seed, and tests intact.
Distinguish generated suggestions from executed results.
After the failure experiment, ask me to explain the mechanism.
```

In [1]:
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression,Ridge,Lasso,LogisticRegression
from sklearn.metrics import mean_squared_error,balanced_accuracy_score
rng = np.random.default_rng(402)
X = rng.normal(size=(220,100)); X[:,1] = X[:,0] + rng.normal(0,.02,220)
y = 3*X[:,0] - 2*X[:,2] + rng.normal(0,1,220)
tr,te = np.arange(65),np.arange(65,220)
models = {'OLS':LinearRegression(),'ridge':Ridge(alpha=20),'lasso':Lasso(alpha=.12),'overshrunk':Ridge(alpha=1e8)}
errors={}; fitted={}
for name,estimator in models.items():
    fitted[name]=make_pipeline(StandardScaler(),estimator).fit(X[tr],y[tr])
    errors[name]=mean_squared_error(y[te],fitted[name].predict(X[te]))
print('Held-out MSE:',errors)
assert errors['lasso'] < errors['OLS'] and errors['overshrunk'] > errors['lasso']


Held-out MSE: {'OLS': 4.1264392698677375, 'ridge': 3.8464944114150716, 'lasso': 1.2565333488446822, 'overshrunk': 13.972977564869108}


In [2]:
for name in ['ridge','lasso']:
    weights=fitted[name][-1].coef_
    print(name,'paired feature weights:',weights[:2], 'nonzero count:',np.count_nonzero(weights))
assert np.count_nonzero(fitted['lasso'][-1].coef_) < X.shape[1]
label=(y>0).astype(int)
clf=make_pipeline(StandardScaler(),LogisticRegression(C=1,max_iter=500)).fit(X[tr],label[tr])
p=clf.predict_proba(X[te])[:,1]
print('Logistic balanced accuracy:',balanced_accuracy_score(label[te],p>=.5))
assert np.all((p>=0)&(p<=1)) and p.shape==(len(te),)


ridge paired feature weights: [0.92396027 0.9099756 ] nonzero count: 100
lasso paired feature weights: [2.42035215 0.        ] nonzero count: 23
Logistic balanced accuracy: 0.7466666666666666


## Deliberate failure and repair

Excessive regularization suppresses the useful signal along with noise. Compare the overshrunk model with the mean-only baseline before claiming improvement. Inspect the correlated pair of coefficients: do not rank anatomical importance solely by their magnitudes. Repair a report by naming scale, penalty, selection rule, and held-out metric.

## Your investigation

Repeat with 30 and 65 training observations, preserving the untouched test set only for this teaching comparison. Explain why a real researcher cannot keep choosing experiments by those test outcomes. Examine lasso stability across bootstrap samples of training participants; does feature selection remain stable when correlated columns substitute for one another?

## Transfer to real neuroimaging

Use a region-by-participant table or an independently specified voxel mask. Preserve feature names and units, and compare imaging models with demographic or task-only baselines where appropriate. Pipeline construction does not cure a mask selected using every participant’s outcome.

**Primary teaching sources, pinned where hosted on GitHub:**

- [BrainIAK: regularization and nested optimization](https://github.com/brainiak/brainiak-tutorials/blob/fb62ede943d9694fe703aee0df5f43ecf5558415/tutorials/05-classifier-optimization.ipynb)
- [NMA: multiple and polynomial regression](https://github.com/NeuromatchAcademy/course-content/blob/44634e960df7a14cd0bf7398187f2d209d26b0e8/tutorials/W1D2_ModelFitting/student/W1D2_Tutorial4.ipynb)

Pinned upstream tutorials are a separate assignment; they have **not been executed** by this core lab. They may require data downloads, specialist dependencies, unfinished student cells, and additional compute.

## Exit questions and answer key

1. Does a zero lasso coefficient establish biological irrelevance? **No; penalty and correlated alternatives can remove it.**
2. Where should penalty selection happen? **Within training data, with inner validation when estimating the complete tuning procedure.**

### Return to the research question

Reopen [PM01](../../curriculum/papers/modeling.md#pm01), [PM03](../../curriculum/papers/modeling.md#pm03) and your initial two-sentence prediction. In your [evidence ledger](../../curriculum/coursework/EVIDENCE_LEDGER.md):

1. Cite one output or diagnostic from this lesson and explain the transformation it demonstrates.
2. Revise one claim or question from the paper, with a figure/section locator. State what this small exercise still cannot establish about the published result.
3. Ask AI to propose a next check. Accept, revise or reject it with a scientific reason. Then explain your decision aloud without reading the AI response.

Reuse this entry in the A2 portfolio when relevant; a separate report is unnecessary.
